# Kalman Historical V2 Nested Candidate

기존 V1 historical run을 보존한 채 별도 V2 candidate를 생성합니다.

- nested walk-forward: outer test untouched
- `__coverage` / near-zero variance / highly correlated redundant features 제거
- maximum features = 32
- strategy 10% / 100% vs buy-and-hold 10% / 100% / volatility-matched benchmark
- Equal Weight / Inverse Vol / HRP / Riskfolio 비교
- Qlib 0.9.7은 Python 3.12 isolated environment에서만 기록

**Safety:** Research only / Toss OFF / Neon write OFF / 기존 V1 artifact overwrite 없음


In [ ]:
from google.colab import drive
import json
import shutil
import subprocess
import traceback
from datetime import datetime
from pathlib import Path

PINNED_SHA = "19c1b489e39b4ce18920bbf7597e567f6d56382a"
SOURCE_RUN_TAG = "20260913_042850"
CANDIDATE_TAG = "20260913_nested_v2_001"

drive.mount('/content/drive', force_remount=False)

diag_dir = Path('/content/drive/MyDrive/Kalman_Diagnostics')
diag_dir.mkdir(parents=True, exist_ok=True)
bootstrap_status = diag_dir / 'historical_v2_candidate_bootstrap_status.json'

def write_status(status, **extra):
    payload = {
        'status': status,
        'updated_at': datetime.now().astimezone().isoformat(),
        'source_run_tag': SOURCE_RUN_TAG,
        'candidate_tag': CANDIDATE_TAG,
        'pinned_sha': PINNED_SHA,
        **extra,
    }
    tmp = bootstrap_status.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str) + '\n', encoding='utf-8')
    tmp.replace(bootstrap_status)

write_status('RUNNING', phase='CLONE')

try:
    repo = Path('/content/Codex')
    if repo.exists():
        shutil.rmtree(repo)

    subprocess.run(
        ['git', 'clone', '--branch', 'main', 'https://github.com/kimtk94/Codex.git', str(repo)],
        check=True,
    )
    write_status('RUNNING', phase='PINNED_CHECKOUT')
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', PINNED_SHA], check=True)
    checked = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
    assert checked == PINNED_SHA, (checked, PINNED_SHA)

    app = repo / 'kalman-toss-gateway'
    runner = app / 'scripts' / 'colab_historical_v2_candidate.py'
    required = [
        runner,
        app / 'research' / 'quant_stack' / 'historical_v2_candidate.py',
        app / 'research' / 'quant_stack' / 'historical_v2_candidate_riskfolio.py',
        app / 'research' / 'quant_stack' / 'historical_v2_candidate_qlib.py',
        app / 'config' / 'model-v2-historical-candidate-spec.json',
    ]
    missing = [str(path) for path in required if not path.exists()]
    assert not missing, missing

    write_status('RUNNING', phase='PY_COMPILE')
    subprocess.run(
        ['python', '-m', 'py_compile', *[str(path) for path in required if path.suffix == '.py']],
        check=True,
    )

    write_status('RUNNING', phase='CANDIDATE_RUNNER')
    subprocess.run(
        [
            'python', str(runner),
            '--drive-root', '/content/drive/MyDrive',
            '--source-run-tag', SOURCE_RUN_TAG,
            '--candidate-tag', CANDIDATE_TAG,
            '--pinned-code-sha', PINNED_SHA,
        ],
        check=True,
    )

    summary = (
        Path('/content/drive/MyDrive/Market_Model_V2/historical_quant_2017_v2_candidate')
        / CANDIDATE_TAG
        / 'historical_v2_candidate_summary.json'
    )
    assert summary.exists(), summary
    write_status('COMPLETE', phase='DONE', summary=str(summary))
    print('\nSUMMARY:', summary)
    print(summary.read_text(encoding='utf-8'))
except Exception as exc:
    write_status(
        'FAIL',
        phase='BOOTSTRAP_OR_RUNNER',
        error_type=type(exc).__name__,
        error=str(exc),
        traceback=traceback.format_exc(),
    )
    raise
